# LangChain Core Components & RAG Implementation with Llama-3

This notebook serves as a comprehensive guide to building LLM-powered applications using the **LangChain** framework and **Groq** (Llama-3-8B). It transitions from basic prompt engineering to complex Retrieval-Augmented Generation (RAG) architectures.

### Tech Stack
* **LLM:** Llama-3.1-8B-Instant (via Groq)
* **Orchestration:** LangChain Expression Language (LCEL)
* **Vector Store:** Chroma DB
* **Embeddings:** HuggingFace (Sentence-Transformers)

### Key Learning Modules
1.  **Foundations:** Initializing Chat Models and managing Chat History.
2.  **Prompt Engineering:** Custom Prompt Templates and `MessagesPlaceholder`.
3.  **Structured Output:** Implementing JSON and CSV Output Parsers.
4.  **Data Ingestion:** PDF & Web loaders with semantic Text Splitting.
5.  **Retrieval Strategy:** Standard Vector Search vs. Advanced **Parent Document Retrieval**.
6.  **RAG Chain:** Building an end-to-end question-answering pipeline.

### Importing required libraries

In [2]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import os
from langchain_groq import ChatGroq

## LLM


In [3]:
from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq model cleanly
llama_llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=256
)

In [3]:
# Test the connection
msg = llama_llm.invoke("Hello, how are you?")
print(msg.content)

I'm functioning properly, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here to help answer any questions or provide information you need. How can I assist you today?


In [5]:
print(llama_llm.invoke("Who is man's best friend?").content)

The phrase "man's best friend" is commonly used to refer to dogs. This term is often associated with the loyalty, companionship, and affection that dogs provide to humans.


### Chat message


The chat model takes a list of messages as input and returns a new message. All messages have both a role and a content property.  Here's a list of the most commonly used types of messages:

- `SystemMessage`: Use this message type to prime AI behavior.  This message type is  usually passed in as the first in a sequence of input messages.
- `HumanMessage`: This message type represents a message from a person interacting with the chat model.
- `AIMessage`: This message type, which can be either text or a request to invoke a tool, represents a message from the chat model.

We can find more message types at [LangChain built-in message types](https://python.langchain.com/v0.2/docs/how_to/custom_chat_model/#messages).


The following code imports the most common message type classes from LangChain:


In [6]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

Now let's create a few messages that simulate a chat experience with the bot:


In [7]:
msg = llama_llm.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)

In [11]:
print(msg)

content='You might enjoy "Gone Girl" by Gillian Flynn, a twisty and suspenseful mystery novel that explores the complexities of marriage and deception.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 66, 'total_tokens': 97, 'completion_time': 0.050265346, 'completion_tokens_details': None, 'prompt_time': 0.00413923, 'prompt_tokens_details': None, 'queue_time': 0.005977237, 'total_time': 0.054404576}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cfc8b-2d4a-7fd3-b5d9-30b1a106a40c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 66, 'output_tokens': 31, 'total_tokens': 97}


The model responded with an `AI` message.


We can use these message types to pass an entire chat history along with the AI's responses to the model:


In [12]:
msg = llama_llm.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

In [13]:
print(msg)

content='Aiming for 3-4 times a week is a good starting point to allow for adequate recovery time and progressive overload.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 87, 'total_tokens': 113, 'completion_time': 0.046172578, 'completion_tokens_details': None, 'prompt_time': 0.004898076, 'prompt_tokens_details': None, 'queue_time': 0.005330378, 'total_time': 0.051070654}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cfc8c-78fe-78f3-8607-3353b0e3af80-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 87, 'output_tokens': 26, 'total_tokens': 113}


We can also exclude the system message.


In [14]:
msg = llama_llm.invoke(
    [
        HumanMessage(content="What month follows June?")
    ]
)

In [15]:
print(msg)

content='The month that follows June is July.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 40, 'total_tokens': 49, 'completion_time': 0.007230444, 'completion_tokens_details': None, 'prompt_time': 0.002680057, 'prompt_tokens_details': None, 'queue_time': 0.006447734, 'total_time': 0.009910501}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cfc8d-1002-7752-8857-09b5679539ec-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 9, 'total_tokens': 49}


### Prompt templates


Prompt templates help translate user input and parameters into instructions for a language model. We can use prompt templates to guide a model's response, helping the model understand the context and generate relevant and coherent language-based output.

Next, explore several different types of prompt templates.


#### String prompt templates


Use these prompt templates to format a single string. These templates are generally used for simpler inputs.


In [9]:
from langchain_core.prompts import PromptTemplate

Then, create a prompt template with variables for customization. We also create a dictionary to store inputs that will replace the placeholders. The keys match the variable names in the template, and values are what will be inserted.


In [17]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {"adjective": "funny", "topic": "cats"}  # create a dictionary to store the corresponding input to placeholders in prompt template

Finally, format the prompt template with the input dictionary. The code below invokes the prompt with our input values, replacing {adjective} with "funny" and {topic} with "cats". The result will be a formatted string: "Tell me one funny joke about cats".


In [18]:
prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

#### Chat prompt templates


We can use these prompt templates to format a list of messages. These "templates" consist of lists of templates.


In [19]:
from langchain_core.prompts import ChatPromptTemplate

In [23]:
# Create a ChatPromptTemplate with a list of message tuples
# Each tuple contains a role ("system" or "user") and the message content
# The system message sets the behavior of the assistant
# The user message includes a variable placeholder {topic} that will be replaced later
prompt = ChatPromptTemplate.from_messages([
 ("system", "You are a helpful assistant"),
 ("user", "Tell me a joke about {topic}")
])

# Create a dictionary with the variable to be inserted into the template
# The key "topic" matches the placeholder name in the user message
input_ = {"topic": "cats"}

# Format the chat template with our input values
# This replaces {topic} with "cats" in the user message
# The result will be a formatted chat message structure ready to be sent to a model
print(prompt.invoke(input_))

messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cats', additional_kwargs={}, response_metadata={})]


####  MessagesPlaceholder


We can use the MessagesPlaceholder prompt template to add a list of messages in a specific location. In `ChatPromptTemplate.from_messages`, we saw how to format two messages, with each message as a string. But what if we want the user to supply a list of messages that we would slot into a particular spot? we can use `MessagesPlaceholder` for this task.


In [25]:
# Import MessagesPlaceholder for including multiple messages in a template
from langchain_core.prompts import MessagesPlaceholder
# Import HumanMessage for creating message objects with specific roles
from langchain_core.messages import HumanMessage

# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# The system message sets the behavior for the assistant
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
("system", "You are a helpful assistant"),
MessagesPlaceholder("msgs")  # This will be replaced with one or more messages
])

# Create an input dictionary where the key matches the MessagesPlaceholder name
# The value is a list of message objects that will replace the placeholder
# Here we're adding a single HumanMessage asking about the day after Tuesday
input_ = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

# Format the chat template with our input dictionary
# This replaces the MessagesPlaceholder with the HumanMessage in our input
# The result will be a formatted chat structure with a system message and our human message
print(prompt.invoke(input_))

messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the day after Tuesday?', additional_kwargs={}, response_metadata={})]


We can wrap the prompt and the chat model and pass them into a chain, which can invoke the message.


In [26]:
chain = prompt | llama_llm
response = chain.invoke(input = input_)
print(response)

content='The day after Tuesday is Wednesday.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 47, 'total_tokens': 55, 'completion_time': 0.014385635, 'completion_tokens_details': None, 'prompt_time': 0.002299998, 'prompt_tokens_details': None, 'queue_time': 0.005496351, 'total_time': 0.016685633}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cfcfb-9cbe-72c1-b767-114d284b5f6c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 8, 'total_tokens': 55}


### Output parsers


Output parsers take the output from an LLM and transform that output to a more suitable format. Parsing the output is very useful when we are using LLMs to generate any form of structured data, or to normalize output from chat models and other LLMs.


LangChain has lots of different types of output parsers. This is a [list](https://python.langchain.com/v0.2/docs/concepts/#output-parsers) of output parsers LangChain supports. We will use the following two output parsers as examples:

- `JSON`: Returns a JSON object as specified. We can specify a Pydantic model and it will return JSON for that model. Probably the most reliable output parser for getting structured data that does NOT use function calling.
- `CSV`: Returns a list of comma separated values.


#### JSON parser


This output parser allows users to specify an arbitrary JSON schema and query LLMs for outputs that conform to that schema.


In [28]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [ ]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [30]:
# And a query intended to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

In [35]:
# Get the formatting instructions for the output parser
# This generates guidance text that tells the LLM how to format its response
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [39]:
# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nQuery: {query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)
print(prompt.invoke({"query": joke_query}).text)

Answer the user query.
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Mar

In [40]:
# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | llama_llm | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to Llama
# 3. Parse the response into the structure defined by our output parser
# 4. Return the structured result
chain.invoke({"query": joke_query})

{'setup': 'Why was the math book sad?',
 'punchline': 'Because it had too many problems.'}

#### Comma-separated list parser


Use the comma-separated list parser when we want a list of comma-separated items.


In [2]:
# Import the CommaSeparatedListOutputParser to parse LLM responses into Python lists
from langchain_core.output_parsers import CommaSeparatedListOutputParser
# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()
# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

In [3]:
format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [10]:
# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],  # This variable will be provided when the chain is invoked
    partial_variables={"format_instructions": format_instructions},  # This variable is set once when creating the prompt
)

# Build a processing chain that:
# 1. Takes the subject and formats it into the prompt template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response into a Python list using the CommaSeparatedListOutputParser
chain = prompt | llama_llm | output_parser

# Invoke the processing chain with "ice cream flavors" as the subject
# This will:
# 1. Substitute "ice cream flavors" into the prompt template
# 2. Send the formatted prompt to the Llama LLM
# 3. Parse the LLM's comma-separated response into a Python list
chain.invoke({"subject": "ice cream flavors"})

['Vanilla',
 'Chocolate',
 'Strawberry',
 'Cookies and Cream',
 'Mint Chocolate Chip']

#### **Creating and Using a JSON Output Parser** 

We'll complete the following steps:

1. Import the necessary components to create a JSON output parser.
2. Create a prompt template that requests information in JSON format (hint: use the provided template).
3. Build a chain that connects our prompt, LLM, and JSON parser.
4. Test our parser using at least three different inputs.
5. Access and display specific fields from the parsed JSON output.
6. Verify that our output is properly structured and accessible as a Python dictionary.

In [4]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [5]:
# Create your JSON parser
json_parser = JsonOutputParser()

# Create the format instructions
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

In [6]:
print(prompt_template.invoke({"movie_name": "The Matrix"}).text)

You are a JSON-only assistant.

Task: Generate info about the movie "The Matrix" in JSON format.

RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON.



In [15]:
# Create the chain
movie_chain = prompt_template | llama_llm | json_parser

# Test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction, Action


### Documents


#### Document object


A `Document` object in `LangChain` contains information about some data. A Document object has the following two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. We can use the metadata to track various details, such as the document ID, the file name, and other details.


Let's examine how to create a `Document` object. `LangChain` uses the  `Document` object type to handle text or documents.


In [4]:
# Import the Document class from langchain_core.documents module
# Document is a container for text content with associated metadata
from langchain_core.documents import Document

# Create a Document instance with:
# 1. page_content: The actual text content about Python
# 2. metadata: A dictionary containing additional information about this document
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
metadata={
    'my_document_id' : 234234,                      # Unique identifier for this document
    'my_document_source' : "About Python",          # Source or title information
    'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
 })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language.\n Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

Note that we don't have to include metadata.


In [5]:
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
Python's design philosophy emphasizes code readability with its notable use of significant indentation.""")

Document(metadata={}, page_content="Python is an interpreted high-level general-purpose programming language.\nPython's design philosophy emphasizes code readability with its notable use of significant indentation.")

#### Document loaders


Document loaders in LangChain are designed to load documents from a variety of sources; for instance, loading a PDF file and having the LLM read the PDF file using LangChain.

LangChain offers over 100 distinct document loaders, along with integrations with other major providers, such as AirByte and Unstructured. These integrations enable loading of all kinds of documents (HTML, PDF, code) from various locations including private Amazon S3 buckets, as well as from public websites).

We can find a list of document types that LangChain can load at [LangChain Document loaders](https://python.langchain.com/v0.1/docs/integrations/document_loaders/).

We will use the PDF loader and the URL and website loader.


##### **PDF loader**


By using the PDF loader, we can load a PDF file as a `Document` object.

We will load the following paper about using LangChain. we can access and read the paper here: [Revolutionizing Mental Health Care through LangChain: A Journey with a Large Language Model](https://doi.org/10.48550/arXiv.2403.05568).


In [6]:
# Import the PyPDFLoader class from langchain_community's document_loaders module
# This loader is specifically designed to load and parse PDF files
from langchain_community.document_loaders import PyPDFLoader

# Create a PyPDFLoader instance by passing the URL of the PDF file
# The loader will download the PDF from the specified URL and prepare it for loading
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

# Call the load() method to:
# 1. Download the PDF if needed
# 2. Extract text from each page
# 3. Create a list of Document objects, one for each page of the PDF
# Each Document will contain the text content of a page and metadata including page number
document = loader.load()

Here, `document` is a `Document` object with `page_content` and `metadata`:


In [7]:
document[2]  # take a look at the page 2

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 2, 'page_label': '3'}, page_content='Figure 2. An AIMessage illustration \nC. Prompt Template \nPrompt templates [10] allow you to structure input for LLMs. \nThey provide a convenient way to format user inputs and \nprovide instructions to generate responses. Prompt templates \nhelp ensure that the LLM understands the desired context and \nproduces relevant outputs. \nThe prompt template classes in LangChain are built to \nmake constructing prompts with dynamic inputs easier. Of \nthese classes, the simplest is the PromptTemplate. \nD. Chain \nChains [11] in LangChain refer to the combination of \nmultiple components to achieve specific tasks. Th

In [8]:
print(document[1].page_content[:1000])  # print the page 1's first 1000 tokens

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionalities encompass: 
1. Context-Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context-aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a few-
shot examples, or existing content, to ground their 
responses effectively. 
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, these appl

##### **URL and website loader**


We can also load content from a URL or website into a `Document` object:


In [9]:
# Import the WebBaseLoader class from langchain_community's document_loaders module
# This loader is designed to scrape and extract text content from web pages
from langchain_community.document_loaders import WebBaseLoader

# Create a WebBaseLoader instance by passing the URL of the web page to load
# This URL points to the LangChain documentation's introduction page
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

In [10]:
# Call the load() method to:
# 1. Send an HTTP request to the specified URL
# 2. Download the HTML content
# 3. Parse the HTML to extract meaningful text
# 4. Create a list of Document objects containing the extracted content
web_data = loader.load()

In [11]:
# Print the first 1000 characters of the page content from the first Document
# This provides a preview of the successfully loaded web content
# web_data[0] accesses the first Document in the list
# .page_content accesses the text content of that Document
# [:1000] slices the string to get only the first 1000 characters
print(web_data[0].page_content[:1000])

LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsUI-Library integrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain is an open source framework with a prebuilt agent architecture and integrations for any model or tool—so you can build agents that adapt as fast as the ecosystem evolvesCopy pageLangChain is the easy way 

#### Text splitters


After we load documents, we will often want to transform those documents to better suit our application.


One of the most simple examples of making documents better suit our application is to split a long document into smaller chunks that can fit into our model's context window. LangChain has built-in document transformers that ease the process of splitting, combining, filtering, and otherwise manipulating documents.

At a high level, here is how text splitters work:

1. They split the text into small, semantically meaningful chunks (often sentences).
2. They start combining these small chunks of text into a larger chunk until we reach a certain size (as measured by a specific function).
3. After the combined text reaches the new chunk's size, make that chunk its own piece of text and then start creating a new chunk of text with some overlap to keep context between chunks.

For a list of types of text splitters LangChain supports, see [LangChain Text Splitters](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/).

Let's use a simple `CharacterTextSplitter` as an example of how to split the LangChain paper we just loaded.

This is the simplest method. This splits based on characters (by default "\n\n") and measures chunk length by number of characters.

`CharacterTextSplitter` is the simplest method of splitting the content. These splits are based on characters (by default "\n\n") and measures chunk length by number of characters.


In [12]:
# Import the CharacterTextSplitter class
# Text splitters are used to divide large texts into smaller, manageable chunks
from langchain_text_splitters import CharacterTextSplitter

In [13]:
# Create a CharacterTextSplitter with specific configuration:
# - chunk_size=200: Each chunk will contain approximately 200 characters
# - chunk_overlap=20: Consecutive chunks will overlap by 20 characters to maintain context
# - separator="\n": Text will be split at newline characters when possible
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

# Split the previously loaded document (PDF or other text) into chunks
# The split_documents method:
# 1. Takes a list of Document objects
# 2. Splits each document's content based on the configured parameters
# 3. Returns a new list of Document objects where each contains a chunk of text
# 4. Preserves the original metadata for each chunk
chunks = text_splitter.split_documents(document)

# Print the total number of chunks created
# This shows how many smaller Document objects were generated from the original document(s)
# The number depends on the original document length and the chunk_size setting
print(len(chunks))

147


The CharacterTextSplitter splits the document into 148 chunks. Let's look at the content of a chunk:


In [14]:
chunks[5].page_content   # take a look at any chunk's page content

'individuals seeking guidance and support in these critical areas. \nMindGuide lever ages the capabilities of LangChain and its \nChatModels, specifically Chat OpenAI, as the bedrock of its'

In [15]:
print(chunks[5].metadata)
print(chunks[6].metadata)

{'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
{'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


#### Working with Document Loaders and Text Splitters

**We will do this:**

1. Import the necessary document loaders to work with both PDF and web content.
2. Load the provided paper about LangChain architecture.
3. Create two different text splitters with varying parameters.
4. Compare the resulting chunks from different splitters.
5. Examine the metadata preservation across splitting.
6. Create a simple function to display statistics about our document chunks.

In [16]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [17]:
# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
print("\n" + "="*50 + "\n")  # Separator for clarity
display_document_stats(chunks_2, "Splitter 2")


=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.80 characters
Metadata keys preserved: creationdate, title, producer, source, creator, moddate, page_label, author, total_pages, page

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Min chunk size: 49 characters
Max chunk size: 299 characters



=== Splitter 2 Statistics ===
Total number of chunks: 57
Average chunk size: 452.74 characters
Metadata keys preserved: creationdate, title, producer, source, creat

#### Embedding models


Embedding models are specifically designed to interface with text embeddings.

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing us to conceptualize text within a vector space. Consequently, we can perform operations such as semantic search, where we identify pieces of text that are most similar within the vector space.


IBM, OpenAI, Hugging Face, and others offer embedding models.

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

# We use a highly efficient open-source sentence transformer
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2" 
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1027.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
# Test the embedding
texts = [chunk.page_content for chunk in chunks_1[:5]]  # Take the first 5 chunks from splitter 1 for testing
embedding_result = embedding_model.embed_documents(texts)

print(f"Embedding result for the first chunk (length {len(embedding_result[0])}):")
for i, embedding in enumerate(embedding_result):
    print(f"Chunk {i+1} embedding (first 5 dimensions): {embedding[:5]}")

Embedding result for the first chunk (length 384):
Chunk 1 embedding (first 5 dimensions): [0.014029835350811481, 0.011181734502315521, 0.02445284090936184, -0.0012005901662632823, -0.06421826779842377]
Chunk 2 embedding (first 5 dimensions): [0.01199568435549736, 0.04280419647693634, -0.0304243266582489, -0.011556822806596756, 0.0014073814963921905]
Chunk 3 embedding (first 5 dimensions): [0.03430046886205673, 0.017374210059642792, 0.019214311614632607, 0.05843944475054741, -0.07801038026809692]
Chunk 4 embedding (first 5 dimensions): [-0.018644466996192932, -0.022325441241264343, 0.06158606335520744, -0.012890773825347424, -0.07388195395469666]
Chunk 5 embedding (first 5 dimensions): [0.00384152727201581, -0.07433420419692993, 0.024721112102270126, -0.010318024083971977, 0.07208865880966187]


#### Vector stores


One of the most common ways to store and search over unstructured data is to embed the text data and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. We can use a [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) to store embedded data and perform vector search for us.


We will use the code uses `Chroma`.


In [20]:
from langchain_chroma import Chroma

Next, have the embedding model perform the embedding process and store the resulting vectors in the Chroma vector database.


In [21]:
docsearch = Chroma.from_documents(chunks, embedding_model)

Then we can use a similarity search strategy to retrieve the information that is related to our query. The model returns a list of similar or relevant document chunks. Here, we can view the code that prints the contents of the most similar chunk.


In [22]:
query = "Langchain"
docs = docsearch.similarity_search(query)
print(len(docs))
print(docs[0].page_content)

4
II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models


#### Retrievers


A retriever is an interface that returns documents using an unstructured query. Retrievers are more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. We can still use vector stores as the backbone of a retriever. Note that other types of retrievers also exist.

Retrievers accept a string `query` as input and return a list of `Documents` as output.


##### **Vector store-backed retrievers**


Vector store retrievers are retrievers that use a vector store to retrieve documents. They are a lightweight wrapper around the vector store class to make it conform to the retriever interface. They use the search methods implemented by a vector store, such as similarity search and MMR (Maximum marginal relevance), to query the texts in the vector store.

Now that we have constructed a vector store `docsearch`, we can easily construct a retriever such as seen in the following code.


In [23]:
# Use the docsearch vector store as a retriever
# This converts the vector store into a retriever interface that can fetch relevant documents
retriever = docsearch.as_retriever()

# Invoke the retriever with the query "Langchain"
# This will:
# 1. Convert the query text "Langchain" into an embedding vector
# 2. Perform a similarity search in the vector store using this embedding
# 3. Return the most semantically similar documents to the query
docs = retriever.invoke("Langchain")

# Access the first (most relevant) document from the retrieval results
# This returns the full Document object including:
# - page_content: The text content of the document
# - metadata: Any associated metadata like source, page numbers, etc.
# The returned document is the one most semantically similar to "Langchain"
docs[0]

Document(id='ca896cae-acfb-4b27-8402-ee051b07c33f', metadata={'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'page_label': '1', 'total_pages': 6, 'page': 0, 'creator': 'Microsoft Word', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'author': 'IEEE', 'creationdate': '2023-12-31T03:50:13+00:00', 'producer': 'PyPDF'}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models')

In [24]:
len(docs)

4

Note that the results are identical to the results we obtained using the similarity search strategy.


##### **Parent document retrievers**


When splitting documents for retrieval, there are often conflicting goals:

- We want small documents so their embeddings can most accurately reflect their meaning. If the documents are too long, then the embeddings can lose meaning.
- We want to have long enough documents to retain the context of each chunk of text.

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data. During retrieval, this retriever first fetches the small chunks, but then looks up the parent IDs for the data and returns those larger documents.


In [25]:
# ---------------------------------------------------------
# 1. The Hybrid Imports (Modern + Classic)
# ---------------------------------------------------------
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import Chroma

In [26]:
# ---------------------------------------------------------
# 2. Define the Dual-Splitter Strategy
# ---------------------------------------------------------
# Parent: Large context chunks (2000 chars) that the LLM will actually read
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
# Child: Small fragments (400 chars) used purely for precise vector searching
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)

In [27]:
# ---------------------------------------------------------
# 3. Initialize the Storage Engines
# ---------------------------------------------------------
# The Vector Database (Stores the small child embeddings)
# Note: 'embedding_model' is our HuggingFace model from the previous step
vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=embedding_model 
)

# The Key-Value Database (Stores the massive parent text blocks)
# Set up an in-memory storage layer for the parent documents
# This will store the larger chunks that provide context, but won't be directly embedded
store = InMemoryStore()

In [28]:
# ---------------------------------------------------------
# 4. Assemble the Classic Retriever
# ---------------------------------------------------------
retriever = ParentDocumentRetriever(
    # The vector store where child document embeddings will be stored and searched
    # This Chroma instance will contain the embeddings for the smaller chunks
    vectorstore=vectorstore,
    # The document store where parent documents will be stored
    # These larger chunks won't be embedded but will be retrieved by ID when needed
    docstore=store,
    # The splitter used to create small chunks (400 chars) for precise vector search
    # These smaller chunks are embedded and used for similarity matching
    child_splitter=child_splitter,
    # The splitter used to create larger chunks (2000 chars) for better context
    # These parent chunks provide more complete information when retrieved
    parent_splitter=parent_splitter,
)

In [29]:
len(pdf_document)

6

In [30]:
# ---------------------------------------------------------
# 5. Ingest Data & Execute
# ---------------------------------------------------------
# This automatically splits, embeds, and links UUIDs behind the scenes.
retriever.add_documents(pdf_document)

In [31]:
# Test the architecture
query = "What is the main architecture of LangChain?"
retrieved_docs = retriever.invoke(query)

# Verify that it pulled the massive Parent document, not a tiny Child fragment
print(f"Total Parent Documents Retrieved: {len(retrieved_docs)}")
if retrieved_docs:
    print(f"Length of Top Match: {len(retrieved_docs[0].page_content)} characters")
    print("-" * 50)
    print(retrieved_docs[0].page_content[:500] + "...\n[CONTINUES]")

Total Parent Documents Retrieved: 3
Length of Top Match: 1966 characters
--------------------------------------------------
LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionaliti...
[CONTINUES]


The following code retrieves and counts the number of parent document IDs stored in the document store


In [32]:
len(list(store.yield_keys()))

16

Next, we verify that the underlying vector store still retrieves the small chunks.


In [33]:
sub_docs = vectorstore.similarity_search("Langchain")

In [34]:
print(sub_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to


In [35]:
sub_docs[0].metadata

{'creationdate': '2023-12-31T03:50:13+00:00',
 'producer': 'PyPDF',
 'moddate': '2023-12-31T03:52:06+00:00',
 'title': 's8329 final',
 'page_label': '2',
 'creator': 'Microsoft Word',
 'page': 1,
 'author': 'IEEE',
 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf',
 'total_pages': 6,
 'doc_id': '338dcafa-3d61-45cf-8bf3-3d55e0387177'}

In [36]:
list(store.yield_keys())

['d5626937-bcde-4c67-b486-5288812c85f8',
 'eeed3e5b-7754-41ce-a76c-53468dc0c603',
 'e2b88931-bad2-44b9-9e1d-dfec872a79bb',
 '38eedb6d-aeac-4c02-bd20-8e2814e523b6',
 '338dcafa-3d61-45cf-8bf3-3d55e0387177',
 'f7dfee77-7aeb-4c73-ae51-12c8d54e5649',
 '0ef3cfa3-6124-4139-bc8c-5a9e5eafe001',
 'bf336c0b-2250-4bed-b9a8-91a0b9b6f104',
 'cb80e74b-8892-438b-bc6d-d0306ed37799',
 '185764fd-89b8-4632-9184-29e518506840',
 '50ff9dd8-1947-4d94-b298-3ce6a2bfa9cc',
 '1184d459-72a7-4e8f-b399-a010db7135df',
 'bea6bd4f-d62d-4052-9028-8b30ec887519',
 '32bf85a6-62ed-4cf6-b61e-5f5678e04463',
 '335a7e73-4893-409f-8861-7aa3003e7fb7',
 'd441758e-1489-44fe-98bd-e2099106f1b9']

And then retrieve the relevant large chunk.


In [37]:
retrieved_docs = retriever.invoke("Langchain")

In [38]:
print(retrieved_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionalities encompass: 
1. Context-Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context-aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a few-
shot examples, or existing content, to ground their 
responses effectively. 
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, these appl

In [39]:
retrieved_docs[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2023-12-31T03:50:13+00:00',
 'author': 'IEEE',
 'moddate': '2023-12-31T03:52:06+00:00',
 'title': 's8329 final',
 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf',
 'total_pages': 6,
 'page': 1,
 'page_label': '2'}

In [40]:
# ---------------------------------------------------------
# 1. Inspecting the Massive Parents (Key-Value Store)
# ---------------------------------------------------------
print("=== PARENT STORE INSPECTION ===")
# Extract all the UUID keys generated by the system
parent_keys = list(store.yield_keys())
print(f"Total Parent Chunks in RAM: {len(parent_keys)}")

=== PARENT STORE INSPECTION ===
Total Parent Chunks in RAM: 16


In [41]:
store.mget(parent_keys[0:2])

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental di

In [42]:
store.mget([parent_keys[0]])

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental di

In [43]:
if parent_keys:
    sample_key = parent_keys[0]
    print(f"Sample UUID: {sample_key}")
    # mget() fetches the raw Document object based on the UUID
    sample_parent = store.mget([sample_key])[0]
    print(f"Parent Payload (First 100 chars): {sample_parent.page_content[:100]}...\n")

Sample UUID: d5626937-bcde-4c67-b486-5288812c85f8
Parent Payload (First 100 chars): * corresponding author - jkim72@kent.edu 
Revolutionizing Mental Health Care through 
LangChain: A J...



In [44]:
# ---------------------------------------------------------
# 2. Inspecting the Tiny Children (Vector Database)
# ---------------------------------------------------------
print("=== CHILD STORE INSPECTION ===")
# .get() rips the entire raw dictionary out of Chroma without doing a search
raw_chroma_data = vectorstore.get()

print(f"Total Child Vectors in DB: {len(raw_chroma_data['ids'])}")

=== CHILD STORE INSPECTION ===
Total Child Vectors in DB: 82


In [45]:
list(raw_chroma_data.keys())

['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']

In [46]:
if raw_chroma_data['ids']:
    print(f"Child Vector ID: {raw_chroma_data['ids'][0]}")
    print(f"Child Payload (First 100 chars): {raw_chroma_data['documents'][0][:100]}...")
    
    # THIS IS THE CRITICAL PROOF: 
    # Notice how the metadata contains the exact UUID of the parent.
    print(f"Child Metadata Link: {raw_chroma_data['metadatas'][0]}")

Child Vector ID: a1783ac3-96a2-4a6f-85bf-f7b13cd0d74e
Child Payload (First 100 chars): * corresponding author - jkim72@kent.edu 
Revolutionizing Mental Health Care through 
LangChain: A J...
Child Metadata Link: {'moddate': '2023-12-31T03:52:06+00:00', 'total_pages': 6, 'creator': 'Microsoft Word', 'doc_id': 'd5626937-bcde-4c67-b486-5288812c85f8', 'creationdate': '2023-12-31T03:50:13+00:00', 'page_label': '1', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'producer': 'PyPDF', 'page': 0, 'author': 'IEEE'}


##### **RetrievalQA**


Here's an example using LangChain's `RetrievalQA`.


In [47]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define the formatting function for the retriever
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Expose and define the exact Prompt you want the LLM to use
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

# 3. Build the LCEL Chain (The Modern "RetrievalQA")
# Notice how your document search retriever is plugged directly in
retriever = docsearch.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llama_llm
    | StrOutputParser()
)

# 4. Execute
query = "what is this paper discussing?"
result = rag_chain.invoke(query)

print(result)

Based on the provided context, it appears that this paper is discussing mental health, specifically the complexities of dealing with mental health challenges and the potential benefits of using the LangChain framework to address these issues.


#### **Building a Simple Retrieval System with LangChain**

We'll implement a simple retrieval system using LangChain's vector store and retriever components to help answer questions based on a document.

1. Import the necessary components for document loading, embedding, and retrieval.
2. Load the provided document about artificial intelligence.
3. Split the document into manageable chunks.
4. Use an embedding model to create vector representations.
5. Create a vector store and a retriever.
6. Implement a simple question-answering system.
7. Test our system with at least 3 different questions.

In [48]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [49]:
# ---------------------------------------------------------
# 2. Ingestion & Splitting
# ---------------------------------------------------------
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [50]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1654.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
# ---------------------------------------------------------
# Vector Store & Retriever Initialization
# ---------------------------------------------------------
vector_store = Chroma.from_documents(chunks, embedding_model)

# We configure the retriever to only return the top 3 results (k=3)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [52]:
def search_documents(query):
    """Search for documents relevant to a query."""
    return retriever.invoke(query)

In [53]:
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query in test_queries:
    print(f"\n{'-'*50}\nQuery: {query}\n{'-'*50}")
    
    # Execute the search
    results = search_documents(query)
    
    print(f"Found {len(results)} relevant documents:")
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}: {doc.page_content[:150]}...")
        print(f"Source: {doc.metadata.get('source', 'Unknown')}")


--------------------------------------------------
Query: What is LangChain?
--------------------------------------------------
Found 3 relevant documents:

Result 1: II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing appli...
Source: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf

Result 2: Off-the-Shelf Chains: LangChain offers pre -configured 
chains, which are structured assemblies of components 
tailored to accomplish specific high -l...
Source: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf

Result 3: D. Chain 
Chains [11] in LangChain refer to the combination of 
multiple components to achieve specific tasks. They provide 
a structured and modular ...
Source: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-pape